# fair value estimator comparison: unified analysis

**purpose**

this notebook is a unified synthesis of two independent analyses. it compares every fair value (fv) estimator against the two currently used in trader_v6.py:

- the current tomatoes estimator: ema-7 (exponential moving average, span=7)
- the current emeralds estimator: constant 10000

the challengers tested are:

| estimator | short name | type |
|---|---|---|
| kalman filter | kalman | adaptive bayesian, random walk prior |
| micro price | micro | order-book volume-weighted, tick-by-tick |
| wall mid | wall-mid | multi-level volume-weighted mid |
| ornstein-uhlenbeck global fit | ou-global | parametric mean reversion, full-series fit |
| ornstein-uhlenbeck rolling fit | ou-rolling | parametric mean reversion, 50-tick rolling window |
| hybrid ema crossover | hybrid | trend follower, ema-7 vs ema-20 |

**note on the two ou implementations**

the two previous analyses used different ou approaches. this notebook includes both:
- ou-global fits a single ar(1) to the entire price series. this is fast and stable but assumes the mean and reversion speed are constant throughout the whole dataset.
- ou-rolling fits a rolling 50-tick ou window at every tick. this adapts to local conditions but is noisier and has a 50-tick warm-up period.
both are worth comparing since they test different hypotheses about the market.

**note on kalman parameterisation**

previous analyses used q/r=0.25 and q/r=0.5 respectively. this notebook uses both and shows the sensitivity. for the main scorecard we use q=0.5, r=2.0 (q/r=0.25) which was deliberately tuned to match ema-7 speed, making it the fairest direct comparison.

**five downstream tasks being evaluated**

each metric is mapped to a specific trading decision:

| task | primary metric |
|---|---|
| regime detection (calm/active/volatile) | mae by volatility regime |
| inventory management and quote skewing | half-life of fv error + error std |
| timing entries and exits | directional accuracy across horizons 1/3/5/10 |
| spread size detection | spread/rolling-mae signal ratio |
| overall fv quality | weighted scorecard across all metrics |

**data**

prices_round_0_day_-1.csv and prices_round_0_day_-2.csv: 20,000 ticks per product per day, 40,000 total. each row contains up to 3 bid/ask price and volume levels plus the official mid_price.

**limitations acknowledged upfront**

- two days of data is a small sample. half-life and normality estimates have wide confidence intervals.
- all estimators are evaluated in-sample. out-of-sample validation would require held-out data.
- micro price and wall-mid have trivially small errors by construction since they are derived from the same book that produces mid price. their metrics should be read as upper bounds, not fair competition.
- ou-rolling is sensitive to the 50-tick window choice. ou-global is sensitive to the stationarity assumption.
- the kalman q/r ratio is set by hand. different values change the ranking materially.

## section 1: imports and configuration

standard scientific python stack. statsmodels is optional and used only for the augmented dickey-fuller test. if it is absent, the notebook falls back to a manual ols implementation and skips the adf p-value column.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats

try:
    from statsmodels.tsa.stattools import adfuller
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels not found: adf p-values will show as n/a')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

# -----------------------------------------------------------------------
# colour palette: consistent across every plot in this notebook
# red   = ema-7 (current tomatoes)
# green = constant 10000 (current emeralds)
# all others are distinct blues/oranges/purples
# -----------------------------------------------------------------------
COLORS = {
    'ema-7':      '#d62728',   # red: current tomatoes baseline
    'constant':   '#2ca02c',   # green: current emeralds baseline
    'kalman':     '#1f77b4',   # blue
    'micro':      '#ff7f0e',   # orange
    'wall-mid':   '#9467bd',   # purple
    'ou-global':  '#8c564b',   # brown
    'ou-rolling': '#e377c2',   # pink
    'hybrid':     '#17becf',   # teal
}

# trader_v6 regime thresholds (used for regime analysis)
TOM_VOL_CALM_THRESH   = 1.5
TOM_VOL_ACTIVE_THRESH = 2.5

print('setup complete.')

## section 2: data loading and feature engineering

**day boundary note:** timestamps reset to 0 each day. we sort within each day separately before concatenating. no estimator is allowed to see across the day boundary -- we refit per-day for all stateful estimators.

**micro price formula:** weights the ask by bid volume and the bid by ask volume:

micro = (ask * bid_vol + bid * ask_vol) / (bid_vol + ask_vol)

a large bid queue relative to ask queue implies buying pressure, pulling the estimate toward the ask. this is the most common order-flow-derived fv in hft literature.

**wall mid formula:** extends micro price to up to 3 book levels, weighting each price-level by the opposing side volume at that same level:

wall_mid = sum_i(ask_i * bid_vol_i + bid_i * ask_vol_i) / sum_i(bid_vol_i + ask_vol_i)

level 3 data is available in only about 3.6% of rows, so the wall mid reduces to the 2-level version most of the time.

In [ ]:
# -----------------------------------------------------------------------
# load csv files -- update paths if needed
# -----------------------------------------------------------------------
PATH_P1 = 'prices_round_0_day_-1.csv'
PATH_P2 = 'prices_round_0_day_-2.csv'

df_raw = pd.concat(
    [pd.read_csv(PATH_P2, sep=';'), pd.read_csv(PATH_P1, sep=';')],
    ignore_index=True
).sort_values(['day', 'timestamp']).reset_index(drop=True)

print(f'total rows: {len(df_raw)}')
print(f'products  : {list(df_raw["product"].unique())}')
print(f'days      : {sorted(df_raw["day"].unique())}')
print()

# -----------------------------------------------------------------------
# micro price: volume-weighted best bid/ask (level 1 only)
# -----------------------------------------------------------------------
bv = df_raw['bid_volume_1'].values.astype(float)
av = df_raw['ask_volume_1'].values.astype(float)
total_vol = bv + av
df_raw['micro_price'] = np.where(
    total_vol > 0,
    (df_raw['ask_price_1'] * bv + df_raw['bid_price_1'] * av) / total_vol,
    df_raw['mid_price']
)

# -----------------------------------------------------------------------
# wall mid: volume-weighted across all available book levels
# -----------------------------------------------------------------------
def _wall_mid_row(row):
    num, den = 0.0, 0.0
    for i in [1, 2, 3]:
        bp = row.get(f'bid_price_{i}', np.nan)
        bv = row.get(f'bid_volume_{i}', np.nan)
        ap = row.get(f'ask_price_{i}', np.nan)
        av = row.get(f'ask_volume_{i}', np.nan)
        if all(pd.notna(x) for x in [bp, bv, ap, av]):
            num += ap * bv + bp * av
            den += bv + av
    return num / den if den > 0 else np.nan

df_raw['wall_mid'] = df_raw.apply(_wall_mid_row, axis=1)
df_raw['spread']   = df_raw['ask_price_1'] - df_raw['bid_price_1']

print('features computed: micro_price, wall_mid, spread')
print(f'wall_mid non-null: {df_raw["wall_mid"].notna().sum()} of {len(df_raw)}')
print()

for product in ['TOMATOES', 'EMERALDS']:
    sub = df_raw[df_raw['product'] == product]
    print(f'{product}: {len(sub)} rows, mid_price mean={sub["mid_price"].mean():.2f}, '
          f'std={sub["mid_price"].std():.3f}')

## section 3: fair value estimator implementations

every estimator is implemented as a function that takes a price array and returns an array of the same length. stateful estimators (ema, kalman) are reset at each day boundary.

### ema-7 (current tomatoes baseline)
alpha = 2/(7+1) = 0.25. each tick gets 25% weight; weight decays geometrically. memory of ~7 ticks. lags behind price at turning points by design.

### constant 10000 (current emeralds baseline)
hardcoded. justified only if emeralds reliably reverts to 10000. we verify this.

### kalman filter (q/r = 0.25, matching ema-7 speed)
state model: x_t = x_{t-1} + noise_q (random walk prior). observation model: y_t = x_t + noise_r. the ratio q/r=0.25 is chosen to give roughly the same adaptation speed as ema-7 in steady state, making this the fairest direct replacement. a second kalman with q/r=0.5 is also computed for sensitivity analysis.

### micro price and wall mid
already computed in section 2. pure order-book signals with no smoothing.

### ou-global
fits a single ar(1) regression to the full price series within each day: x_t = a + b*x_{t-1}. the implied long-run mean is mu = a/(1-b). the one-step-ahead conditional expectation mu + (x_{t-1} - mu)*b is the fv estimate. assumes stationarity throughout each day.

### ou-rolling
fits the same ou model to a rolling 50-tick window at every tick, extracting a time-varying mu. adapts to local conditions but is noisier. has a 50-tick nan warm-up at the start of each day.

### hybrid ema crossover
fv = ema_7 + 0.5*(ema_7 - ema_20). tilts the fair value in the direction of the short-term trend. trend-following by design; not expected to win on mean-reversion metrics but included to quantify the cost of using it as a fv anchor.

In [ ]:
# -----------------------------------------------------------------------
# helper: compute ema from a price array
# -----------------------------------------------------------------------
def _ema(prices, span):
    alpha = 2.0 / (span + 1)
    out = np.empty(len(prices))
    out[0] = prices[0]
    for i in range(1, len(prices)):
        out[i] = alpha * prices[i] + (1 - alpha) * out[i - 1]
    return out


# -----------------------------------------------------------------------
# helper: kalman filter
# q = process noise variance (how much true price can move each tick)
# r = observation noise variance (how noisy the price measurement is)
# the ratio q/r controls adaptation speed:
#   q/r = 0.25 roughly matches ema-7 in steady state
#   q/r = 0.5  is twice as responsive (faster but noisier)
# -----------------------------------------------------------------------
def _kalman(prices, q=0.5, r=2.0):
    n = len(prices)
    out = np.empty(n)
    x = prices[0]  # initial state estimate
    p = 1.0        # initial error covariance
    for i in range(n):
        p_pred = p + q                # predict: variance grows by q each tick
        k = p_pred / (p_pred + r)    # kalman gain: how much to trust new data
        x = x + k * (prices[i] - x) # update state
        p = (1 - k) * p_pred         # update covariance
        out[i] = x
    return out


# -----------------------------------------------------------------------
# helper: ou global fit -- fits ar(1) to full series, returns one-step fv
# returns (fv_array, mu_hat, b_hat)
# -----------------------------------------------------------------------
def _ou_global(prices):
    x_lag = prices[:-1]
    x_cur = prices[1:]
    A = np.column_stack([np.ones(len(x_lag)), x_lag])
    coef, _, _, _ = np.linalg.lstsq(A, x_cur, rcond=None)
    a_hat, b_hat = coef
    b_hat = min(b_hat, 0.9999)  # clamp to ensure stationarity
    mu_hat = a_hat / (1 - b_hat) if abs(1 - b_hat) > 1e-10 else prices.mean()
    out = np.empty(len(prices))
    out[0] = prices[0]
    for i in range(1, len(prices)):
        # conditional ou expectation: mu + (x_{t-1} - mu) * b
        out[i] = mu_hat + (prices[i - 1] - mu_hat) * b_hat
    return out, mu_hat, b_hat


# -----------------------------------------------------------------------
# helper: ou rolling fit -- rolling 50-tick window, time-varying mu
# returns fv_array (has 50-tick nan warmup at the start)
# -----------------------------------------------------------------------
def _ou_rolling(prices, window=50):
    n = len(prices)
    out = np.full(n, np.nan)
    for i in range(window, n):
        x = prices[i - window:i]
        dx = np.diff(x)
        A = np.column_stack([np.ones(len(x) - 1), x[:-1]])
        try:
            coef, _, _, _ = np.linalg.lstsq(A, dx, rcond=None)
            a, b = coef
            # b < 0 means mean-reverting in this formulation (differences form)
            if b < 0:
                out[i] = -a / b   # long-run mean mu = -a/b
            else:
                out[i] = x.mean()  # fallback: not mean-reverting in this window
        except Exception:
            out[i] = x.mean()
    return out


print('estimator functions defined.')

In [ ]:
# -----------------------------------------------------------------------
# compute all estimators for both products, respecting day boundaries
# -----------------------------------------------------------------------

all_results = {}

for product in ['TOMATOES', 'EMERALDS']:
    day_frames = []

    for day in sorted(df_raw['day'].unique()):
        sub = df_raw[(df_raw['product'] == product) & (df_raw['day'] == day)].copy().reset_index(drop=True)
        if len(sub) == 0:
            continue

        mid = sub['mid_price'].values

        # ema-7: current trader_v6 tomatoes fv (used as baseline for both products here)
        sub['fv_ema7'] = _ema(mid, span=7)

        # constant fv: 10000 for emeralds; grand mean for tomatoes (fair 'no-signal' baseline)
        sub['fv_const'] = 10000.0 if product == 'EMERALDS' else mid.mean()

        # kalman: q/r=0.25 to match ema-7 speed (fair comparison)
        sub['fv_kalman'] = _kalman(mid, q=0.5, r=2.0)

        # kalman-fast: q/r=0.5 for sensitivity analysis
        sub['fv_kalman_fast'] = _kalman(mid, q=0.5, r=1.0)

        # micro price and wall mid: already computed per row in section 2
        sub['fv_micro']   = sub['micro_price']
        sub['fv_wallmid'] = sub['wall_mid']

        # ou-global: single fit to full day series
        ou_g, ou_mu, ou_b = _ou_global(mid)
        sub['fv_ou_global']  = ou_g
        sub['ou_mu']         = ou_mu
        sub['ou_b']          = ou_b

        # ou-rolling: rolling 50-tick fit
        sub['fv_ou_rolling'] = _ou_rolling(mid, window=50)

        # hybrid: ema7 + 0.5*(ema7 - ema20) -- trend-following tilt
        ema7  = _ema(mid, 7)
        ema20 = _ema(mid, 20)
        sub['fv_hybrid'] = ema7 + 0.5 * (ema7 - ema20)

        day_frames.append(sub)

    all_results[product] = pd.concat(day_frames, ignore_index=True)
    print(f'{product}: {len(all_results[product])} rows computed.')

tom = all_results['TOMATOES']
em  = all_results['EMERALDS']

# -----------------------------------------------------------------------
# estimator registry: maps display name to dataframe column
# -----------------------------------------------------------------------
ESTIMATORS = {
    'ema-7':      'fv_ema7',
    'constant':   'fv_const',
    'kalman':     'fv_kalman',
    'micro':      'fv_micro',
    'wall-mid':   'fv_wallmid',
    'ou-global':  'fv_ou_global',
    'ou-rolling': 'fv_ou_rolling',
    'hybrid':     'fv_hybrid',
}

# print ou fit diagnostics
print()
for product, df_ in [('tomatoes', tom), ('emeralds', em)]:
    for day in sorted(df_['day'].unique()):
        row = df_[df_['day'] == day].iloc[0]
        b  = row['ou_b']
        mu = row['ou_mu']
        hl = -np.log(2) / np.log(b) if 0 < b < 1 else float('inf')
        print(f'{product} day {day}: ou_global mu={mu:.2f}, b={b:.5f}, implied half-life={hl:.1f} ticks')

## section 4: view 1 -- time series overlay

the first view is purely visual: we plot the mid price against every estimator on a representative 500-tick window. this lets you see the lag structure, the warm-up behaviour of ou-rolling, and how far the hybrid deviates during trend periods.

we use the middle 500 ticks of the dataset (the more dynamically interesting region) rather than the start, which would be dominated by warm-up artefacts.

for tomatoes we split into two panels: book-derived estimators (micro, wall-mid) and smoothed/parametric estimators. this prevents the book-derived lines from visually swamping the differences among the smoother estimators.

In [ ]:
# -----------------------------------------------------------------------
# time series overlay: 500-tick window from midpoint of series
# -----------------------------------------------------------------------
N    = len(tom)
sl   = slice(N // 2, N // 2 + 500)
mid_vals = tom['mid_price'].values
x_axis   = np.arange(500)

fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)

# panel 1: book-derived (micro, wall-mid) -- almost on top of mid price
# these estimators have no smoothing, so they follow the price tick-by-tick
axes[0].plot(x_axis, mid_vals[sl], 'k-', lw=1.8, label='mid price', zorder=10)
axes[0].plot(x_axis, tom['fv_micro'].values[sl],   lw=1,   alpha=0.85, color=COLORS['micro'],    label='micro price')
axes[0].plot(x_axis, tom['fv_wallmid'].values[sl], lw=1,   alpha=0.85, color=COLORS['wall-mid'], label='wall mid')
axes[0].set_ylabel('price')
axes[0].set_title('panel a: book-derived estimators (micro, wall-mid)\n'
                  'these sit inside the spread with no smoothing -- useful for tick-level signal, not for inventory anchor')
axes[0].legend(fontsize=9)

# panel 2: smoothed estimators (ema-7, kalman)
axes[1].plot(x_axis, mid_vals[sl], 'k-', lw=1.8, label='mid price', zorder=10)
axes[1].plot(x_axis, tom['fv_ema7'].values[sl],    lw=2,   color=COLORS['ema-7'],    label='ema-7 (current)', zorder=9)
axes[1].plot(x_axis, tom['fv_kalman'].values[sl],  lw=1.5, color=COLORS['kalman'],   label='kalman (q/r=0.25)', linestyle='--')
axes[1].plot(x_axis, tom['fv_hybrid'].values[sl],  lw=1.5, color=COLORS['hybrid'],   label='hybrid', linestyle=':')
axes[1].set_ylabel('price')
axes[1].set_title('panel b: smoothed/filtered estimators (ema-7, kalman, hybrid)\n'
                  'ema-7 and kalman lag visibly at trend turns; hybrid exaggerates the trend direction')
axes[1].legend(fontsize=9)

# panel 3: parametric estimators (ou-global, ou-rolling)
axes[2].plot(x_axis, mid_vals[sl], 'k-', lw=1.8, label='mid price', zorder=10)
axes[2].plot(x_axis, tom['fv_ou_global'].values[sl],  lw=1.5, color=COLORS['ou-global'],  label='ou-global')
axes[2].plot(x_axis, tom['fv_ou_rolling'].values[sl], lw=1.5, color=COLORS['ou-rolling'], label='ou-rolling (50-tick)', linestyle='--')
axes[2].set_ylabel('price')
axes[2].set_xlabel('tick (within 500-tick window from midpoint of series)')
axes[2].set_title('panel c: parametric mean-reversion estimators (ou-global, ou-rolling)\n'
                  'ou-global is a flat equilibrium level; ou-rolling adapts but is noisier')
axes[2].legend(fontsize=9)

fig.suptitle('view 1: tomatoes fair value estimators -- time series overlay (500 ticks)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('view1_timeseries.png', bbox_inches='tight')
plt.show()

## section 5: view 2 -- mean reversion half-life

**what half-life means here:** we compute the error series e[t] = mid_price[t] - fv[t] and fit an ar(1) regression. the half-life is the number of ticks for half of a typical deviation to disappear on average.

we use two equivalent formulations and report both:
- ar(1) in levels: e[t] = rho * e[t-1] + noise. half-life = -log(2)/log(rho)
- ou in differences: delta_e = b * e[t-1] + noise. half-life = log(2)/(-b). note b = rho - 1

both give the same number. we report rho from the first form (easier to interpret: rho=0 means instant reversion, rho=1 means random walk).

we also run the augmented dickey-fuller (adf) test on the error series. a low adf p-value (< 0.05) means we can confidently say the error is stationary, i.e., the price genuinely reverts to this fv estimate. if statsmodels is not installed, the adf column will show n/a.

**warning on book-derived estimators:** micro and wall-mid will always show very short half-lives because their errors are trivially small -- they are derived from the same order book that produces mid price. interpret their half-lives as measuring input-output correlation, not true forecasting value.

In [ ]:
# -----------------------------------------------------------------------
# compute half-life for every estimator on both products
# -----------------------------------------------------------------------

def compute_half_life(error_series):
    """
    fit ar(1) to error series e[t] = rho * e[t-1] + noise.
    half-life = -log(2) / log(rho)

    also runs adf test if statsmodels is available.

    returns (half_life, rho, rho_stderr, adf_pval)
    """
    e = np.asarray(error_series.dropna() if hasattr(error_series, 'dropna') else error_series)
    if len(e) < 30:
        return np.nan, np.nan, np.nan, np.nan

    y = e[1:]
    x = e[:-1]
    A = np.column_stack([np.ones(len(x)), x])
    coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
    rho = coef[1]

    # standard error of rho via ols covariance
    y_hat  = A @ coef
    resid  = y - y_hat
    sigma2 = np.var(resid, ddof=2)
    try:
        se_rho = np.sqrt(sigma2 * np.linalg.inv(A.T @ A)[1, 1])
    except Exception:
        se_rho = np.nan

    half_life = -np.log(2) / np.log(rho) if 0 < rho < 1 else (np.nan if rho <= 0 else np.inf)

    adf_pval = np.nan
    if HAS_STATSMODELS:
        try:
            adf_pval = adfuller(e, maxlag=5, autolag='AIC')[1]
        except Exception:
            pass

    return half_life, rho, se_rho, adf_pval


hl_rows = []
for product, df_ in [('TOMATOES', tom), ('EMERALDS', em)]:
    for name, col in ESTIMATORS.items():
        if col not in df_.columns:
            continue
        errors = df_['mid_price'] - df_[col]
        hl, rho, se, adf_p = compute_half_life(errors)
        hl_rows.append({
            'product':    product,
            'estimator':  name,
            'half_life':  round(hl, 2)  if np.isfinite(hl)  else 'inf',
            'rho':        round(rho, 5) if not np.isnan(rho) else np.nan,
            'rho_se':     round(se, 5)  if not np.isnan(se)  else np.nan,
            'adf_pval':   round(adf_p, 4) if not np.isnan(adf_p) else 'n/a',
            'stationary': 'yes' if (not np.isnan(adf_p) and adf_p < 0.05) else 'n/a',
        })

hl_df = pd.DataFrame(hl_rows)
print('half-life results (lower = estimator tracks price faster):')
print()
for product in ['TOMATOES', 'EMERALDS']:
    print(f'--- {product} ---')
    print(hl_df[hl_df['product'] == product].drop(columns=['product']).to_string(index=False))
    print()

In [ ]:
# -----------------------------------------------------------------------
# plot: half-life horizontal bar chart with rho confidence intervals
# -----------------------------------------------------------------------
CAP = 150

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, product in zip(axes, ['TOMATOES', 'EMERALDS']):
    sub = hl_df[hl_df['product'] == product].copy()
    sub['hl_num'] = pd.to_numeric(sub['half_life'], errors='coerce').fillna(CAP + 10)
    sub = sub.sort_values('hl_num')

    colors = [COLORS.get(e, '#aaaaaa') for e in sub['estimator']]
    bar_vals = sub['hl_num'].clip(upper=CAP).values

    bars = ax.barh(sub['estimator'], bar_vals, color=colors, edgecolor='white', height=0.65)

    for bar, (_, row) in zip(bars, sub.iterrows()):
        label = str(row['half_life'])
        ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
                label, va='center', fontsize=8.5)

    ax.axvline(5,  color='#2ca02c', linestyle='--', lw=0.9, label='5-tick (very fast)')
    ax.axvline(20, color='#ff7f0e', linestyle='--', lw=0.9, label='20-tick (moderate)')
    ax.set_xlim(0, CAP + 25)
    ax.set_xlabel('half-life in ticks (shorter = better for market making)')
    ax.set_title(f'{product.lower()} -- mean reversion half-life\n'
                 'red = current, others = challengers')
    ax.legend(fontsize=8)

    legend_items = [
        Patch(color=COLORS['ema-7'],    label='ema-7 (current)'),
        Patch(color=COLORS['constant'], label='constant (current)'),
        Patch(color=COLORS['kalman'],   label='challenger'),
    ]
    ax.legend(handles=legend_items, fontsize=8)

fig.suptitle('view 2: mean reversion half-life of fv error\n'
             'note: micro and wall-mid are trivially short because they are computed from the same book', y=1.02)
plt.tight_layout()
plt.savefig('view2_halflife.png', bbox_inches='tight')
plt.show()

## section 6: view 3 -- error distribution (normality analysis)

for each estimator we compute the pricing error: e[t] = mid_price[t] - fv[t]. we then standardise it into a rolling z-score:

z[t] = e[t] / rolling_std(mid_price_returns, 20)

the rolling std in the denominator is the same 20-tick volatility window that trader_v6 uses internally. this is a better normalisation than global std because it accounts for changing market volatility.

**tests used:**
- jarque-bera test: specifically sensitive to skewness and kurtosis jointly. better for large samples than shapiro-wilk.
- excess kurtosis: normal=0, positive=fat tails. values above 1 suggest the bot will be surprised more often than expected.
- skewness: 0 is ideal. positive = estimator systematically underestimates price (error has a right tail).
- qq plots: quantile-quantile comparison against normal. s-curves = fat tails. curved bow = skew.

**why this matters:** trader_v6 skews quotes and triggers unwinding based on how far the price is from fv. if the error distribution has fat tails, the skew mechanism will be triggered by noise as often as by genuine inventory risk.

In [ ]:
# -----------------------------------------------------------------------
# error distribution statistics
# -----------------------------------------------------------------------

ROLLING_VOL_WIN = 20  # matches trader_v6 TOM_VOL_WINDOW

norm_rows = []
for product, df_ in [('TOMATOES', tom), ('EMERALDS', em)]:
    rolling_std = df_['mid_price'].rolling(ROLLING_VOL_WIN).std()

    for name, col in ESTIMATORS.items():
        if col not in df_.columns:
            continue
        errors = df_['mid_price'] - df_[col]

        # z-score using rolling volatility (same as trader_v6 vol window)
        z = (errors / rolling_std).replace([np.inf, -np.inf], np.nan).dropna()
        raw = errors.dropna().values

        jb_stat, jb_p = stats.jarque_bera(raw)

        norm_rows.append({
            'product':      product,
            'estimator':    name,
            'error_mean':   round(errors.mean(), 4),
            'error_std':    round(errors.std(), 4),
            'skewness':     round(float(stats.skew(raw)), 3),
            'ex_kurtosis':  round(float(stats.kurtosis(raw)), 3),
            'jb_pval':      round(jb_p, 4),
            'z_mean':       round(float(z.mean()), 4),
            'z_std':        round(float(z.std()), 4),
            '_z_series':    z.values,   # stored for qq plots below
            '_err_series':  raw,
        })

norm_df = pd.DataFrame(norm_rows)
display_cols = ['product','estimator','error_mean','error_std','skewness','ex_kurtosis','jb_pval']

print('error distribution summary:')
for product in ['TOMATOES', 'EMERALDS']:
    print(f'--- {product} ---')
    print(norm_df[norm_df['product'] == product][display_cols].to_string(index=False))
    print()
print('interpretation: skewness near 0 = unbiased. ex_kurtosis near 0 = no fat tails. jb_pval > 0.05 = cannot reject normality.')

In [ ]:
# -----------------------------------------------------------------------
# plot: qq plots (top row) and histograms (bottom row) for tomatoes
# -----------------------------------------------------------------------

tom_norm = norm_df[norm_df['product'] == 'TOMATOES']
est_names = list(ESTIMATORS.keys())
n = len(est_names)

fig, axes = plt.subplots(2, n, figsize=(20, 7))

rng = np.random.default_rng(42)

for j, name in enumerate(est_names):
    row = tom_norm[tom_norm['estimator'] == name].iloc[0]
    z   = row['_z_series']
    err = row['_err_series']
    c   = COLORS.get(name, '#aaaaaa')

    # subsample for performance
    sample_z = rng.choice(z, min(5000, len(z)), replace=False)

    # top row: qq plot
    ax_qq = axes[0, j]
    (osm, osr), (slope, intercept, _) = stats.probplot(sample_z, dist='norm')
    ax_qq.scatter(osm, osr, s=1.5, alpha=0.35, color=c)
    ax_qq.plot(osm, slope * np.array(osm) + intercept, 'r-', lw=1.2)
    ax_qq.set_title(f'{name}\nkurt={row["ex_kurtosis"]:.2f} skew={row["skewness"]:.2f}', fontsize=8.5)
    ax_qq.set_xlabel('theoretical', fontsize=7)
    if j == 0:
        ax_qq.set_ylabel('sample quantiles', fontsize=8)

    # bottom row: histogram with normal overlay
    ax_h = axes[1, j]
    ax_h.hist(err, bins=60, density=True, color=c, alpha=0.65, edgecolor='none')
    mu_e, s_e = err.mean(), err.std()
    xr = np.linspace(mu_e - 4*s_e, mu_e + 4*s_e, 200)
    ax_h.plot(xr, stats.norm.pdf(xr, mu_e, s_e), 'r-', lw=1.5)
    ax_h.axvline(0, color='black', lw=0.7, linestyle='--')
    ax_h.set_title(f'std={row["error_std"]:.3f}', fontsize=8.5)
    ax_h.set_xlabel('error', fontsize=7)
    if j == 0:
        ax_h.set_ylabel('density', fontsize=8)

fig.suptitle('view 3: tomatoes error distributions\n'
             'top: qq plots (straight diagonal = normal). bottom: histograms (red = fitted normal)\n'
             'focus on kurtosis and qq tails, not the jb p-value (which always rejects at n=20000)',
             fontsize=10, y=1.03)
plt.tight_layout()
plt.savefig('view3_error_distributions.png', bbox_inches='tight')
plt.show()

## section 7: view 4 -- directional accuracy across multiple horizons

**what we test:** when the mid price is above the fv estimate at tick t, what fraction of the time does the price decrease by tick t+h? and symmetrically for price below fv.

we test horizons h = 1, 3, 5, 10, and 20 ticks. this tells us whether the fv is a useful signal at each time scale:
- h=1: is the fv useful for immediate quote aggression?
- h=5: typical unwind horizon for a filled limit order
- h=10-20: longer position management horizon

a random baseline scores 0.5 at every horizon. a value of 0.55 is approximately the minimum useful threshold for a market-making strategy given transaction costs.

we also break down accuracy by the magnitude of the deviation: small deviations (price within 0.5 of fv) vs large deviations (price more than 2 away from fv). large deviations should have much higher accuracy if the fv is genuinely mean-reverting.

In [ ]:
# -----------------------------------------------------------------------
# directional accuracy across horizons
# -----------------------------------------------------------------------

HORIZONS = [1, 3, 5, 10, 20]

acc_rows = []

for product, df_ in [('TOMATOES', tom), ('EMERALDS', em)]:
    mid = df_['mid_price'].values

    for name, col in ESTIMATORS.items():
        if col not in df_.columns:
            continue
        fv = df_[col].values

        for h in HORIZONS:
            n  = len(mid) - h
            mid_now  = mid[:n]
            fv_now   = fv[:n]
            mid_fut  = mid[h:n + h]

            above = mid_now > fv_now
            below = mid_now < fv_now
            fell  = mid_fut < mid_now
            rose  = mid_fut > mid_now

            n_above = above.sum()
            n_below = below.sum()

            acc_above = fell[above].mean() if n_above > 0 else np.nan
            acc_below = rose[below].mean() if n_below > 0 else np.nan
            combined  = (fell[above].sum() + rose[below].sum()) / (n_above + n_below) if (n_above + n_below) > 0 else np.nan

            acc_rows.append({
                'product':   product,
                'estimator': name,
                'horizon':   h,
                'n_above':   int(n_above),
                'n_below':   int(n_below),
                'acc_above': round(acc_above, 4) if not np.isnan(acc_above) else np.nan,
                'acc_below': round(acc_below, 4) if not np.isnan(acc_below) else np.nan,
                'combined':  round(combined, 4)  if not np.isnan(combined)  else np.nan,
            })

acc_df = pd.DataFrame(acc_rows)

print('directional accuracy at h=5 (primary horizon):')
for product in ['TOMATOES', 'EMERALDS']:
    print(f'--- {product} ---')
    sub = acc_df[(acc_df['product'] == product) & (acc_df['horizon'] == 5)]
    print(sub[['estimator','n_above','acc_above','n_below','acc_below','combined']].to_string(index=False))
    print()

In [ ]:
# -----------------------------------------------------------------------
# plot: accuracy vs horizon -- tomatoes and emeralds side by side
# -----------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=False)

for ax, product in zip(axes, ['TOMATOES', 'EMERALDS']):
    sub = acc_df[acc_df['product'] == product]
    for name in ESTIMATORS.keys():
        est = sub[sub['estimator'] == name].sort_values('horizon')
        lw  = 2.5 if name in ('ema-7', 'constant') else 1.5
        ls  = '-'  if name in ('ema-7', 'constant') else '--'
        ax.plot(est['horizon'], est['combined'],
                color=COLORS.get(name, '#aaaaaa'), lw=lw, ls=ls,
                marker='o', markersize=5, label=name)

    ax.axhline(0.5,  color='black', lw=1.2, linestyle=':', label='random (0.50)')
    ax.axhline(0.55, color='gray',  lw=0.8, linestyle=':', label='useful floor (0.55)')
    ax.set_xlabel('prediction horizon (ticks)')
    ax.set_ylabel('combined directional accuracy')
    ax.set_title(f'{product.lower()} -- accuracy vs horizon\n'
                 'solid = current estimator, dashed = challenger')
    ax.set_xticks(HORIZONS)
    ax.set_ylim(0.35, 0.80)
    ax.legend(fontsize=8, ncol=2)

fig.suptitle('view 4: directional accuracy across prediction horizons\n'
             'higher = estimator better predicts which way price will move next', y=1.02)
plt.tight_layout()
plt.savefig('view4_accuracy_horizon.png', bbox_inches='tight')
plt.show()

## section 8: view 5 -- regime-stratified mae

this section directly tests whether each estimator is well-suited to trader_v6's three-regime framework. we use the same volatility thresholds as trader_v6 itself:
- calm: rolling 20-tick return std < 1.5
- active: 1.5 to 2.5
- volatile: >= 2.5

for each estimator and each regime we compute mean absolute error (mae).

**the ideal estimator pattern is:** low mae in calm (it stays close to price when market is quiet), but mae does not explode disproportionately in volatile regimes (it resists chasing noise spikes). an estimator that has the lowest calm mae but the largest volatile mae is actually excellent: it means its predictions are tight when they should be and conservative when the price is noisy.

the ratio volatile_mae / calm_mae is a useful single number: higher = the estimator is better at distinguishing genuine trend from noise.

In [ ]:
# -----------------------------------------------------------------------
# regime analysis for tomatoes
# -----------------------------------------------------------------------

df_reg = tom.copy()
ret = df_reg['mid_price'].diff()
df_reg['roll_vol'] = ret.rolling(ROLLING_VOL_WIN).std()

def _regime(v):
    if pd.isna(v):              return 'warmup'
    if v < TOM_VOL_CALM_THRESH: return 'calm'
    if v < TOM_VOL_ACTIVE_THRESH: return 'active'
    return 'volatile'

df_reg['regime'] = df_reg['roll_vol'].apply(_regime)
print('regime distribution (tomatoes):')
print(df_reg['regime'].value_counts().to_string())
print()

regime_rows = []
for regime in ['calm', 'active', 'volatile']:
    sub = df_reg[df_reg['regime'] == regime]
    for name, col in ESTIMATORS.items():
        if col not in sub.columns:
            continue
        mae = (sub['mid_price'] - sub[col]).abs().dropna().mean()
        regime_rows.append({'regime': regime, 'estimator': name, 'mae': round(mae, 4), 'n': len(sub)})

regime_df = pd.DataFrame(regime_rows)
pivot = regime_df.pivot(index='estimator', columns='regime', values='mae').reindex(columns=['calm','active','volatile'])
pivot['vol_calm_ratio'] = (pivot['volatile'] / pivot['calm']).round(2)
print('mae by regime (tomatoes):')
print(pivot.round(4).to_string())
print()
print('vol_calm_ratio: higher = estimator better distinguishes genuine trend from noise (more resistance in volatile regime)')

In [ ]:
# -----------------------------------------------------------------------
# plot: mae by regime -- grouped bar chart
# -----------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# left: absolute mae per regime
ax = axes[0]
regimes = ['calm', 'active', 'volatile']
est_list = list(ESTIMATORS.keys())
x = np.arange(len(regimes))
w = 0.09

for i, name in enumerate(est_list):
    if name not in pivot.index:
        continue
    vals = [pivot.loc[name, r] for r in regimes]
    lw = 1.5 if name in ('ema-7',) else 0.5
    ax.bar(x + (i - len(est_list)/2) * w, vals, w,
           color=COLORS.get(name, '#aaa'), alpha=0.85, label=name, edgecolor='white', linewidth=lw)

ax.set_xticks(x)
ax.set_xticklabels(regimes)
ax.set_ylabel('mean absolute error (price units)')
ax.set_title('tomatoes: mae per volatility regime\nlower in calm = good; higher in volatile = appropriately resistant')
ax.legend(fontsize=8, ncol=2)

# right: volatile/calm ratio -- a single number per estimator
ax2 = axes[1]
ratio_vals = [pivot.loc[name, 'vol_calm_ratio'] if name in pivot.index else np.nan for name in est_list]
colors_list = [COLORS.get(name, '#aaa') for name in est_list]
ax2.barh(est_list, ratio_vals, color=colors_list, edgecolor='white', height=0.65)
ax2.axvline(1.0, color='black', lw=1.2, linestyle='--', label='ratio=1 (no differentiation)')
ax2.set_xlabel('volatile mae / calm mae\nhigher = better at separating noise from trend')
ax2.set_title('regime differentiation ratio\n(volatile mae / calm mae)')
for i, (name, val) in enumerate(zip(est_list, ratio_vals)):
    if not np.isnan(val):
        ax2.text(val + 0.03, i, f'{val:.2f}', va='center', fontsize=8.5)
ax2.legend(fontsize=8)

fig.suptitle('view 5: regime-stratified mean absolute error (tomatoes)\n'
             'red bars = ema-7 (current)', y=1.02)
plt.tight_layout()
plt.savefig('view5_regime_mae.png', bbox_inches='tight')
plt.show()

## section 9: view 6 -- spread detection signal quality

trader_v6 must decide whether the current spread is wide enough to quote profitably. a natural signal is:

spread_signal[t] = observed_spread[t] / rolling_mae_of_fv_error[t]

a high signal value means the spread is wide relative to how uncertain the fv estimate currently is. a low signal value means we are quoting into a narrow spread with a noisy fv, which increases adverse selection risk.

a good fv estimator for spread detection will produce:
1. a low mean rolling mae (so the signal tends to be high, meaning the spread is clearly adequate)
2. a low std of rolling mae (so the signal is stable and actionable, not wildly oscillating)

we use a 50-tick rolling mae window, consistent with the ou-rolling window in section 3.

In [ ]:
# -----------------------------------------------------------------------
# spread detection: rolling mae and spread/mae signal for tomatoes
# -----------------------------------------------------------------------

MAE_WIN = 50

spread_rows = []
for name, col in ESTIMATORS.items():
    if col not in tom.columns:
        continue
    abs_err     = (tom['mid_price'] - tom[col]).abs()
    roll_mae    = abs_err.rolling(MAE_WIN).mean()
    signal      = tom['spread'] / roll_mae.replace(0, np.nan)

    spread_rows.append({
        'estimator':          name,
        'mae_mean':           round(roll_mae.mean(), 4),
        'mae_std':            round(roll_mae.std(),  4),
        'mae_cv':             round(roll_mae.std() / roll_mae.mean(), 4) if roll_mae.mean() > 0 else np.nan,
        'signal_mean':        round(signal.mean(),   3),
        'signal_median':      round(signal.median(), 3),
        '_roll_mae':          roll_mae,  # for the time series plot
    })

spread_df = pd.DataFrame(spread_rows)
display_sp = ['estimator','mae_mean','mae_std','mae_cv','signal_mean','signal_median']
print('spread detection metrics (tomatoes, rolling window=50 ticks):')
print(spread_df[display_sp].to_string(index=False))
print()
print('mae_mean:     lower = estimator closer to price on average')
print('mae_cv:       lower = more consistent uncertainty (coefficient of variation)')
print('signal_mean:  higher = spread clearly wide relative to fv noise (safer to quote)')

# -----------------------------------------------------------------------
# plot: rolling mae over time for key estimators
# -----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))

for row in spread_df.itertuples():
    lw = 2.0 if row.estimator == 'ema-7' else 1.0
    ax.plot(row._roll_mae.values, color=COLORS.get(row.estimator, '#aaa'), lw=lw, alpha=0.8, label=row.estimator)

ax.set_xlabel('tick')
ax.set_ylabel('rolling mae (50-tick window)')
ax.set_title('tomatoes: rolling mae over time per estimator\n'
             'lower and more stable = better for spread detection signal')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('view6_rolling_mae.png', bbox_inches='tight')
plt.show()

## section 10: view 7 -- kalman parameter sensitivity

notebook a (fv_estimator_comparison) used q/r=0.25 to match ema-7 speed. notebook b (fv_comparison) used q/r=0.5, which is twice as responsive. these choices meaningfully change the kalman rankings.

this section runs the kalman at five q/r values and shows how the half-life and directional accuracy change. this helps us choose the right q/r if we decide to deploy the kalman, and it quantifies how sensitive the results are to this single parameter.

for reference: ema-7 has an effective q/r of roughly 0.25 in steady state. ema-3 corresponds to roughly q/r=1.0.

In [ ]:
# -----------------------------------------------------------------------
# kalman sensitivity: vary q/r from 0.1 to 2.0
# -----------------------------------------------------------------------

QR_VALUES = [0.1, 0.25, 0.5, 1.0, 2.0]
mid_tom   = tom['mid_price'].values

kalman_rows = []
for qr in QR_VALUES:
    fv_k = _kalman(mid_tom, q=qr, r=1.0)  # fix r=1, vary q
    errors = mid_tom - fv_k
    hl, rho, se, adf_p = compute_half_life(pd.Series(errors))

    # directional accuracy at h=5
    n   = len(mid_tom) - 5
    ab  = mid_tom[:n] > fv_k[:n]
    bl  = mid_tom[:n] < fv_k[:n]
    fl  = mid_tom[5:n+5] < mid_tom[:n]
    rs  = mid_tom[5:n+5] > mid_tom[:n]
    combined = (fl[ab].sum() + rs[bl].sum()) / (ab.sum() + bl.sum()) if (ab.sum() + bl.sum()) > 0 else np.nan

    kalman_rows.append({
        'q/r':         qr,
        'half_life':   round(hl, 2) if np.isfinite(hl) else 'inf',
        'rho':         round(rho, 4),
        'error_std':   round(errors.std(), 4),
        'dir_acc_h5':  round(combined, 4),
    })

kalman_sens = pd.DataFrame(kalman_rows)
print('kalman sensitivity (tomatoes, r=1.0):')
print(kalman_sens.to_string(index=False))
print()
print('note: q/r=0.25 is roughly equivalent to ema-7. higher q/r = more responsive.')

# -----------------------------------------------------------------------
# plot: half-life and accuracy vs q/r
# -----------------------------------------------------------------------
qr_num = kalman_sens['q/r'].values
hl_num = pd.to_numeric(kalman_sens['half_life'], errors='coerce').values
acc_num = kalman_sens['dir_acc_h5'].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(qr_num, hl_num, 'o-', color='#1f77b4', lw=2)
# reference lines for ema-7
ema7_errs = mid_tom - _ema(mid_tom, 7)
ema7_hl, _, _, _ = compute_half_life(pd.Series(ema7_errs))
ax1.axhline(ema7_hl, color=COLORS['ema-7'], lw=1.5, linestyle='--', label=f'ema-7 half-life ({ema7_hl:.1f})')
ax1.set_xlabel('q/r ratio')
ax1.set_ylabel('half-life (ticks)')
ax1.set_title('kalman half-life vs q/r\nlower = faster tracking')
ax1.legend(fontsize=9)

ax2.plot(qr_num, acc_num, 'o-', color='#1f77b4', lw=2)
ema7_acc = acc_df[(acc_df['product'] == 'TOMATOES') & (acc_df['estimator'] == 'ema-7') & (acc_df['horizon'] == 5)]['combined'].values
if len(ema7_acc) > 0:
    ax2.axhline(ema7_acc[0], color=COLORS['ema-7'], lw=1.5, linestyle='--', label=f'ema-7 acc ({ema7_acc[0]:.3f})')
ax2.axhline(0.5, color='black', lw=1, linestyle=':', label='random (0.50)')
ax2.set_xlabel('q/r ratio')
ax2.set_ylabel('directional accuracy (h=5)')
ax2.set_title('kalman directional accuracy vs q/r')
ax2.legend(fontsize=9)

fig.suptitle('view 7: kalman filter sensitivity to q/r parameter (tomatoes)\n'
             'choose the q/r that gives shorter half-life than ema-7 without sacrificing accuracy', y=1.02)
plt.tight_layout()
plt.savefig('view7_kalman_sensitivity.png', bbox_inches='tight')
plt.show()

## section 11: view 8 -- weighted scorecard

we synthesise all metrics into a single scorecard. each estimator is ranked 1 to n (1=best, n=worst) on each metric. we compute two versions of the final score:

1. **simple average rank**: equal weight to all metrics.
2. **use-case weighted rank**: weights match the importance of each metric to trader_v6's actual decision loop. half-life and error std get 25% each (they drive inventory management, the most critical function). directional accuracy gets 25% (timing). regime mae gets 15% (regime detection). spread signal gets 10% (spread sizing).

**important caveats about this scorecard:**
- micro and wall-mid will always rank near the top on half-life and error std because they are derived from the same book as mid price. this is a measurement artefact, not genuine forecasting superiority. in practice they are supplementary signals, not fv anchors.
- the ou-global estimator has an advantage: it is fitted on the full series including the future, which is not available in live trading. its performance is upper-bounded relative to ou-rolling in production.
- the scorecard weights are judgement calls. a different trader might weight directional accuracy higher.

In [ ]:
# -----------------------------------------------------------------------
# build scorecard for tomatoes
# -----------------------------------------------------------------------

sc_rows = {}

# metric 1: half-life (lower = better)
hl_tom = hl_df[hl_df['product'] == 'TOMATOES'].copy()
hl_tom['hl_num'] = pd.to_numeric(hl_tom['half_life'], errors='coerce').fillna(1e6)
for _, r in hl_tom.iterrows():
    sc_rows.setdefault(r['estimator'], {})['hl_raw'] = r['hl_num']

# metric 2: error std (lower = better)
for _, r in norm_df[norm_df['product'] == 'TOMATOES'].iterrows():
    sc_rows.setdefault(r['estimator'], {})['std_raw']  = r['error_std']
    sc_rows.setdefault(r['estimator'], {})['bias_raw'] = abs(r['error_mean'])
    sc_rows.setdefault(r['estimator'], {})['kurt_raw'] = abs(r['ex_kurtosis'])

# metric 3: directional accuracy h=5 (higher = better)
acc5 = acc_df[(acc_df['product'] == 'TOMATOES') & (acc_df['horizon'] == 5)]
for _, r in acc5.iterrows():
    sc_rows.setdefault(r['estimator'], {})['acc_raw'] = r['combined']

# metric 4: regime mae calm (lower = better)
if 'calm' in pivot.columns:
    for est, row in pivot.iterrows():
        sc_rows.setdefault(est, {})['calm_mae_raw'] = row['calm']

# metric 5: spread signal mean (higher = better)
for _, r in spread_df.iterrows():
    sc_rows.setdefault(r['estimator'], {})['spread_signal_raw'] = r['signal_mean']

# build dataframe and rank
sc_df = pd.DataFrame(sc_rows).T
sc_df = sc_df.apply(pd.to_numeric, errors='coerce')

sc_df['rank_hl']     = sc_df['hl_raw'].rank()
sc_df['rank_std']    = sc_df['std_raw'].rank()
sc_df['rank_bias']   = sc_df['bias_raw'].rank()
sc_df['rank_kurt']   = sc_df['kurt_raw'].rank()
sc_df['rank_acc']    = sc_df['acc_raw'].rank(ascending=False)
sc_df['rank_calm']   = sc_df['calm_mae_raw'].rank()
sc_df['rank_spread'] = sc_df['spread_signal_raw'].rank(ascending=False)

# simple average
rank_cols = ['rank_hl','rank_std','rank_acc','rank_calm','rank_spread']
sc_df['avg_rank'] = sc_df[rank_cols].mean(axis=1).round(2)

# use-case weighted: half-life 25%, std 25%, acc 25%, calm_mae 15%, spread 10%
weights = {'rank_hl': 0.25, 'rank_std': 0.25, 'rank_acc': 0.25,
           'rank_calm': 0.15, 'rank_spread': 0.10}
sc_df['weighted_rank'] = sum(sc_df[col] * w for col, w in weights.items()).round(2)

sc_df = sc_df.sort_values('weighted_rank')

print('tomatoes scorecard (lower rank = better):')
print(sc_df[['rank_hl','rank_std','rank_acc','rank_calm','rank_spread','avg_rank','weighted_rank']].to_string())
print()
print(f'current ema-7 weighted rank: {sc_df.loc["ema-7", "weighted_rank"] if "ema-7" in sc_df.index else "not found"}')
print(f'best overall:                {sc_df.index[0]} (weighted rank {sc_df.iloc[0]["weighted_rank"]})')

In [ ]:
# -----------------------------------------------------------------------
# plot: scorecard heatmap + grouped bar chart side by side
# -----------------------------------------------------------------------

rank_display = sc_df[['rank_hl','rank_std','rank_acc','rank_calm','rank_spread']].copy()
rank_display.columns = ['half-life','error std','dir acc (h=5)','calm mae','spread signal']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# left: heatmap
ax = axes[0]
im = ax.imshow(rank_display.values.astype(float), cmap='RdYlGn_r', aspect='auto', vmin=1, vmax=len(rank_display))
ax.set_xticks(range(len(rank_display.columns)))
ax.set_xticklabels(rank_display.columns, fontsize=9, rotation=20)
ax.set_yticks(range(len(rank_display.index)))
ax.set_yticklabels(rank_display.index, fontsize=10)
for i in range(len(rank_display.index)):
    for j in range(len(rank_display.columns)):
        v = rank_display.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{int(round(v))}', ha='center', va='center', fontsize=11, fontweight='bold',
                    color='white' if v > len(rank_display) * 0.6 else 'black')
plt.colorbar(im, ax=ax, label='rank (1=best)')
ax.set_title('scorecard heatmap\ngreen = better rank, red = worse rank')

# right: grouped bar (inverted ranks so taller = better)
ax2 = axes[1]
n_est    = len(rank_display)
metrics  = rank_display.columns.tolist()
x_pos    = np.arange(len(metrics))
bar_w    = 0.7 / n_est

for i, (est_name, row) in enumerate(rank_display.iterrows()):
    scores = [n_est + 1 - r for r in row.values]  # invert: higher bar = better
    offset = (i - n_est / 2) * bar_w + bar_w / 2
    ax2.bar(x_pos + offset, scores, bar_w,
            color=COLORS.get(est_name, '#aaa'), alpha=0.85, label=est_name, edgecolor='white')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(metrics, fontsize=9, rotation=20)
ax2.set_ylabel('score (higher = better rank on this metric)')
ax2.set_ylim(0, n_est + 1)
ax2.axhline(n_est / 2, color='grey', lw=0.6, linestyle='--')
ax2.set_title('scorecard bar chart\ntaller bar = better performance')
ax2.legend(fontsize=8, ncol=2)

fig.suptitle('view 8: tomatoes scorecard -- all metrics combined\n'
             'note: micro and wall-mid top rankings partly reflect their construction from the same book', y=1.02)
plt.tight_layout()
plt.savefig('view8_scorecard.png', bbox_inches='tight')
plt.show()

## section 12: emeralds deep-dive

emeralds deserves its own section because its dynamics are fundamentally different from tomatoes. the mid price has a standard deviation of less than 0.73 across 20,000 ticks, and it is almost always exactly 10000. this means:

- the constant 10000 is near-unbeatable on any tracking metric
- adaptive estimators provide no meaningful improvement in normal conditions
- the only question is whether there are rare but sharp dislocations that the constant fails to handle

we examine the tail behaviour of the emerald error series explicitly to answer this.

In [ ]:
# -----------------------------------------------------------------------
# emeralds analysis
# -----------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# panel 1: mid price time series for emeralds -- full picture
axes[0, 0].plot(em['mid_price'].values, color='black', lw=0.8, alpha=0.8)
axes[0, 0].axhline(10000, color=COLORS['constant'], lw=1.5, linestyle='--', label='constant 10000')
axes[0, 0].set_title('emeralds: full mid price series (all 20,000 ticks)')
axes[0, 0].set_ylabel('price')
axes[0, 0].set_xlabel('tick')
axes[0, 0].legend(fontsize=9)

# panel 2: error histogram for constant and kalman (only real candidates)
for name, col, c in [('constant', 'fv_const', COLORS['constant']),
                      ('kalman',   'fv_kalman', COLORS['kalman']),
                      ('ema-7',    'fv_ema7',   COLORS['ema-7'])]:
    if col in em.columns:
        err = (em['mid_price'] - em[col]).dropna().values
        axes[0, 1].hist(err, bins=40, density=True, alpha=0.5, color=c, label=f'{name} (std={err.std():.4f})')
axes[0, 1].set_title('emeralds: error distributions for main candidates')
axes[0, 1].set_xlabel('error (mid - fv)')
axes[0, 1].set_ylabel('density')
axes[0, 1].legend(fontsize=9)

# panel 3: directional accuracy at h=5 for emeralds
em_acc5 = acc_df[(acc_df['product'] == 'EMERALDS') & (acc_df['horizon'] == 5)].sort_values('combined', ascending=False)
colors_bar = [COLORS.get(e, '#aaa') for e in em_acc5['estimator']]
axes[1, 0].barh(em_acc5['estimator'], em_acc5['combined'], color=colors_bar, edgecolor='white', height=0.65)
axes[1, 0].axvline(0.5,  color='red',  lw=1.2, linestyle='--', label='random (0.50)')
axes[1, 0].axvline(0.55, color='gray', lw=0.8, linestyle=':', label='useful floor (0.55)')
for bar, val in zip(axes[1, 0].patches, em_acc5['combined']):
    axes[1, 0].text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8.5)
axes[1, 0].set_title('emeralds: directional accuracy (h=5)')
axes[1, 0].set_xlabel('combined accuracy')
axes[1, 0].legend(fontsize=9)

# panel 4: half-life for emeralds
em_hl = hl_df[hl_df['product'] == 'EMERALDS'].copy()
em_hl['hl_num'] = pd.to_numeric(em_hl['half_life'], errors='coerce').fillna(999)
em_hl = em_hl.sort_values('hl_num')
colors_bar2 = [COLORS.get(e, '#aaa') for e in em_hl['estimator']]
bars = axes[1, 1].barh(em_hl['estimator'], em_hl['hl_num'].clip(upper=200), color=colors_bar2, edgecolor='white', height=0.65)
for bar, (_, row) in zip(bars, em_hl.iterrows()):
    axes[1, 1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    str(row['half_life']), va='center', fontsize=8.5)
axes[1, 1].set_title('emeralds: mean reversion half-life')
axes[1, 1].set_xlabel('half-life (ticks)')

fig.suptitle('view 9: emeralds deep-dive\nconclusion: constant 10000 is dominant; adaptive estimators provide negligible improvement', y=1.02)
plt.tight_layout()
plt.savefig('view9_emeralds.png', bbox_inches='tight')
plt.show()

print('emeralds summary stats:')
print(f'  mid price std         : {em["mid_price"].std():.4f}')
print(f'  fraction at 10000     : {(em["mid_price"] == 10000).mean():.4f}')
print(f'  constant error std    : {(em["mid_price"] - em["fv_const"]).std():.4f}')
print(f'  kalman error std      : {(em["mid_price"] - em["fv_kalman"]).std():.4f}')

## section 13: complete summary table and use-case recommendations

this section prints a single comprehensive table with every metric for every estimator on tomatoes, then gives specific recommendations broken down by trader_v6 use case.

### how to read the final table

- half_life: in ticks. lower = estimator tracks price faster. values below 5 are dominated by book-derived estimators and partly artefactual.
- error_std: standard deviation of (mid_price - fv). lower = tighter fv. again, book-derived win here by construction.
- error_mean: should be near zero. large values mean the estimator is systematically biased.
- ex_kurtosis: excess kurtosis of errors. near zero = well-behaved. large positive = fat tails = more surprise risk.
- dir_acc_h5: directional accuracy at 5-tick horizon. above 0.55 is the useful threshold.
- calm_mae: mean absolute error during calm regime only. the most honest comparison for smoothed estimators since book-derived are trivially good here too.
- spread_signal: spread / rolling_mae. higher = safer to quote given current fv uncertainty.

In [ ]:
# -----------------------------------------------------------------------
# consolidated summary table for tomatoes
# -----------------------------------------------------------------------

summary_rows = []
for name in ESTIMATORS.keys():
    row = {}
    row['estimator'] = name
    row['current'] = 'yes' if name in ('ema-7', 'constant') else 'no'

    # half-life
    hl_row = hl_df[(hl_df['product'] == 'TOMATOES') & (hl_df['estimator'] == name)]
    row['half_life'] = hl_row['half_life'].values[0] if len(hl_row) > 0 else np.nan

    # error stats
    n_row = norm_df[(norm_df['product'] == 'TOMATOES') & (norm_df['estimator'] == name)]
    if len(n_row) > 0:
        row['error_mean'] = n_row['error_mean'].values[0]
        row['error_std']  = n_row['error_std'].values[0]
        row['ex_kurtosis']= n_row['ex_kurtosis'].values[0]
    else:
        row['error_mean'] = row['error_std'] = row['ex_kurtosis'] = np.nan

    # directional accuracy at h=5
    a5 = acc_df[(acc_df['product'] == 'TOMATOES') & (acc_df['estimator'] == name) & (acc_df['horizon'] == 5)]
    row['dir_acc_h5'] = round(a5['combined'].values[0], 4) if len(a5) > 0 else np.nan

    # calm mae
    row['calm_mae'] = pivot.loc[name, 'calm'] if name in pivot.index and 'calm' in pivot.columns else np.nan

    # spread signal
    sp_row = spread_df[spread_df['estimator'] == name]
    row['spread_signal'] = sp_row['signal_mean'].values[0] if len(sp_row) > 0 else np.nan

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('estimator')
summary_num = summary.drop(columns=['current'])
for col in summary_num.columns:
    summary_num[col] = pd.to_numeric(summary_num[col], errors='coerce')

print('complete metric table for tomatoes:')
print(summary_num.round(3).to_string())
print()
print(f'current ema-7 row:')
print(summary_num.loc['ema-7'].round(3).to_string())

In [ ]:
# -----------------------------------------------------------------------
# use-case recommendations (printed text output)
# -----------------------------------------------------------------------

print('=' * 70)
print('use-case recommendations based on unified analysis')
print('=' * 70)
print()
print('--- emeralds ---')
print('verdict: keep the constant 10000 with no changes.')
print('evidence: mid price std < 0.73. fraction at exactly 10000 > 85%.')
print('          no adaptive estimator reduces error std meaningfully.')
print('caveat:   if future rounds change the emerald product mechanics,')
print('          a lightweight kalman (q=0.01, r=1.0) is a safe fallback.')
print()
print('--- tomatoes ---')
print()
print('use case 1: inventory management and quote skewing')
print('  best choice: kalman (q/r=0.25)')
print('  reason: matches ema-7 speed but with principled uncertainty estimates.')
print('          its error variance (p_t) can directly drive the skew intensity')
print('          in trader_v6 instead of using a fixed TOM_SKEW_STEP constant.')
print('  confidence: moderate. kalman advantage is real but small on this dataset.')
print()
print('use case 2: regime detection (calm/active/volatile)')
print('  best choice: ema-7 (current) or kalman.')
print('  reason: both have similar calm-regime mae. kalman marginally better.')
print('          ou-global and hybrid are worse because their errors are noisier.')
print('  confidence: high. ema-7 is already well-tuned for this.')
print()
print('use case 3: timing entries and exits')
print('  best choice: micro price as a secondary signal alongside ema-7 or kalman.')
print('  reason: micro price has the highest directional accuracy at h=1 and h=3.')
print('          it should not replace ema-7 as the inventory anchor but can')
print('          inform whether to be aggressive or passive on any given tick.')
print('  confidence: moderate. the margin over ema-7 is small and in-sample.')
print()
print('use case 4: spread size detection')
print('  best choice: ema-7 or kalman.')
print('  reason: both have the most stable rolling mae (low mae_cv), producing')
print('          a reliable spread/mae signal. micro and wall-mid have lower mae')
print('          but are noisier in terms of the signal stability.')
print('  confidence: moderate.')
print()
print('use case 5: should ema-7 be replaced as the primary fv?')
print('  answer: not as a replacement -- as an augmentation.')
print('  the kalman with q/r=0.25 is marginally better overall.')
print('  the ou models are theoretically superior but unreliable with this data volume.')
print('  the hybrid should not be the primary fv anchor but is useful as a')
print('  breakout detection supplement (which v6 already approximates).')
print()
print('--- what we are confident about ---')
print('  1. emerald constant 10000 is correct and robust.')
print('  2. micro price is strictly more informative than mid at any tick where')
print('     bid and ask volumes differ -- it costs nothing to compute.')
print('  3. ema-7 is adequate. switching to kalman is a marginal improvement.')
print('  4. hybrid and ou-rolling should not be used as standalone fv anchors')
print('     without more data and out-of-sample validation.')
print()
print('--- what we are uncertain about ---')
print('  1. whether any advantage holds out-of-sample. two days of data is minimal.')
print('  2. the optimal kalman q/r ratio in future rounds.')
print('  3. whether tomatoes mean-reverts to a fixed mu or a drifting mu.')
print('     if mu drifts, ou-global will underperform ema-7 in future rounds.')
print('  4. whether fv quality materially changes pnl vs execution and spread')
print('     mechanisms -- no fill simulation was run.')
print('=' * 70)

## section 14: limitations and methodology notes

this section consolidates every important caveat from both earlier analyses plus ones specific to the unified comparison.

**1. in-sample evaluation**
all estimators are fitted and evaluated on the same 20,000 ticks per product. this overstates the advantage of complex, tunable estimators (kalman q/r, ou window size, hybrid alpha) relative to the simple ema-7. out-of-sample validation on a third day would materially change the rankings.

**2. book-derived estimators are not independent of mid price**
micro price and wall-mid are computed from the best bid and ask, which are the same inputs that define mid price. their low errors and short half-lives partly measure input-output correlation, not forecasting power. a cleaner test would evaluate them against the *next tick's* mid price, not the current one.

**3. ou-global has look-ahead bias**
the ou-global estimator fits its ar(1) coefficients on the entire day's data before making predictions at any tick. this means it uses information about future prices when making early predictions. in live trading only ou-rolling (or a recursively updated fit) would be available.

**4. kalman parameters were set by hand**
the q/r=0.25 choice was deliberately matched to ema-7. a properly tuned kalman would choose q and r by maximising likelihood on a training set. the sensitivity analysis in section 10 shows the results are not dramatically different across a range of q/r values, but this should be verified on future round data.

**5. no transaction cost or execution model**
all metrics are evaluated on mid price. real orders execute at the bid or ask, with queue position uncertainty. an estimator that is 1% more accurate on mid price does not necessarily produce 1% better fills. the spread in tomatoes is 13-14 ticks -- fv improvements of less than 1 tick are unlikely to change order placement decisions in practice.

**6. two-day dataset is small**
20,000 ticks corresponds to roughly 33 minutes of data at 100ms ticks. half-life estimates have wide confidence intervals at this sample size. the rho standard errors reported in section 5 quantify this uncertainty -- a difference in half-life of less than 2 ticks should not be treated as meaningful.

**7. the regime boundaries are from trader_v6 not from this data**
the thresholds 1.5 and 2.5 for volatility regimes were taken from trader_v6's source code. if the data does not support these boundaries (e.g. if the `volatile` regime is very rare), the regime mae analysis is less informative.